# 8. 텍스트 수집, 전처리와 토큰화

- 목표: 텍스트 정제, 정규표현식, 토큰화, 형태소 분석, 서브워드, 전통적 인코딩까지 텍스트마이닝의 전처리 축을 정리합니다.
- 흐름: 다음 차시에서 BoW, TF-IDF, 임베딩으로 표현 방식까지 연결합니다.


# 텍스트 데이터와 임베딩

### 과정 소개
자연어 처리(NLP)에서 가장 중요한 출발점은 **텍스트 데이터를 기계가 이해할 수 있는 수치형 데이터로 바꾸는 과정**입니다.  
사람에게는 "언어"가 자연스럽지만, 기계에게는 단순한 문자열일 뿐입니다.  
따라서 언어의 의미를 잃지 않으면서 벡터로 바꾸는 기술, 즉 **임베딩(Embedding)** 은 NLP의 핵심입니다. 

## 1. 자연어 처리 소개

### 1.1 자연어 처리란?
- 컴퓨터가 인간의 언어를 이해·생성·활용할 수 있도록 하는 기술  
- 예: 번역기, 챗봇, 음성 비서, 검색 엔진  

### 1.2 왜 중요한가?
- 오늘날 대부분의 데이터는 **비정형 데이터** (텍스트, 이미지, 음성)  
- 텍스트 데이터는 고객 리뷰, 뉴스, 논문 등 **의사결정에 직결되는 정보**를 담고 있음  
- 기계학습/딥러닝이 성공적으로 작동하려면 이 텍스트를 숫자로 잘 표현해야 함  

### 1.3 변천사
- 규칙 기반 → 통계적 모델 → 신경망 기반 → 임베딩+RNN → Transformer  
- 이번 강의는 **임베딩과 RNN까지** 다루며, 다음 장에서 Transformer로 확장 예정  



## 2. 텍스트 데이터 다루기

### 2.1 텍스트 데이터 수집: 웹 크롤링

텍스트마이닝은 분석할 텍스트를 확보하는 일에서 시작합니다. 직접 설문을 만들 수도 있지만, 뉴스·리뷰·게시글처럼 이미 웹에 공개된 텍스트를 수집해서 분석하는 경우도 많습니다.

`requests + BeautifulSoup`은 HTML을 직접 내려받아 파싱할 때 좋고, Selenium은 브라우저 조작이 필요할 때 익숙한 선택지입니다. 한 단계 더 실무적으로는 **Playwright**를 사용해 브라우저 동작을 기록하고, 기다림과 선택자를 안정적으로 다룰 수 있습니다.

데이터 분석 흐름에서 크롤링은 다음 단계의 가장 앞에 있습니다.

`수집 -> 정제 -> 토큰화 -> 벡터화 -> 모델링`

<img src="image/web_crawling_background.svg" width="780">

이미지 출처: 김민수 강사


#### 웹을 구성하는 요소

**웹(Web)** 은 브라우저와 서버가 정해진 약속에 따라 문서와 데이터를 주고받는 구조입니다. 사용자는 브라우저에서 페이지를 보는 것처럼 느끼지만, 실제로는 여러 개의 요청과 응답이 오갑니다.

| 요소 | 역할 | 크롤링에서 보는 지점 |
|------|------|----------------------|
| 브라우저 | URL을 요청하고 응답을 화면으로 그림 | 개발자도구로 HTML과 요청을 확인함 |
| URL | 웹 자원의 주소 | 도메인, 경로, 쿼리 파라미터를 바꿔 페이지를 이동함 |
| DNS | 도메인 이름을 서버 주소로 바꿈 | 보통 직접 다루지는 않지만, 도메인이 서버를 가리킨다는 배경이 됨 |
| HTTP/HTTPS | 요청과 응답을 주고받는 약속 | 상태 코드, 헤더, 파라미터를 확인함 |
| 웹 서버 | 요청을 받고 HTML이나 데이터를 돌려줌 | 같은 URL을 파이썬 코드로 요청함 |
| HTML | 페이지의 내용과 구조 | 제목, 표, 링크, 날짜가 들어 있는 태그를 찾음 |
| CSS | 화면의 스타일 | 클래스 이름이 선택자 힌트가 되기도 함 |
| JavaScript | 화면 동작과 추가 데이터 요청 | 데이터가 늦게 나타나는 동적 페이지인지 판단함 |
| DOM | 브라우저가 HTML을 트리 구조로 해석한 결과 | `select()`로 원하는 노드를 찾음 |
| API / XHR | 화면 일부를 채우는 별도 데이터 요청 | HTML 대신 JSON 응답을 직접 가져올 수 있음 |

예를 들어 `https://finance.naver.com/item/sise_day.naver?code=005930&page=1`에서 `code=005930`은 종목 코드, `page=1`은 페이지 번호입니다. 크롤링에서는 이런 파라미터를 바꿔가며 여러 페이지를 수집합니다.

HTTP 요청과 응답은 다음처럼 나눠 볼 수 있습니다.

| 구분 | 포함되는 정보 | 확인 이유 |
|------|---------------|-----------|
| Request Method | `GET`, `POST` | 주소만 바꾸면 되는지, 요청 본문이 필요한지 판단함 |
| Query String | `code=005930&page=1` | 검색어, 페이지 번호, 종목 코드처럼 바꿀 값을 찾음 |
| Headers | `User-Agent`, `Referer`, `Cookie` 등 | 서버가 요청을 어떻게 해석하는지 확인함 |
| Status Code | `200`, `404`, `403`, `500` 등 | 요청 성공, 페이지 없음, 접근 거부, 서버 오류를 구분함 |
| Response Body | HTML 또는 JSON | 실제로 파싱할 원본 데이터임 |


#### HTML과 DOM 읽기

HTML은 웹 페이지의 뼈대입니다. 태그가 중첩되면서 문서 구조를 만들고, 각 태그 안에 텍스트나 링크가 들어갑니다.

```html
<tr class="price-row">
  <td class="date">2026.05.17</td>
  <td class="close">70,000</td>
  <td class="volume">12,345,678</td>
</tr>
```

위 HTML에서 `tr`은 표의 한 행, `td`는 표의 한 칸입니다. `class="date"`처럼 태그에 붙은 정보는 **속성(attribute)** 입니다.

Playwright는 브라우저가 해석한 DOM에서 원하는 요소를 `locator`로 찾습니다.

```python
rows = page.locator("tr.price-row")
first_row = rows.nth(0)
date = await first_row.locator("td.date").inner_text()
close = await first_row.locator("td.close").inner_text()
```

크롤링은 결국 화면 안에서 **반복되는 구조**를 찾는 일입니다. 뉴스 목록은 보통 `li`, `article`, `tr` 같은 태그가 반복되고, 표 데이터는 `table -> tr -> td` 구조로 들어 있습니다.


#### 개발자도구로 수집 위치 찾기

크롤링 코드를 바로 작성하기보다 브라우저 개발자도구로 먼저 확인하면 시행착오가 줄어듭니다. Chrome 기준으로는 페이지에서 마우스 오른쪽 클릭 후 **검사**를 누르면 됩니다.

확인할 곳은 주로 두 군데입니다.

1. **Elements 탭**  
   화면에 보이는 글자가 HTML의 어느 태그에 들어 있는지 확인합니다. 제목, 날짜, 표의 행처럼 반복되는 단위를 찾고, 태그 이름·class·id를 봅니다.

2. **Network 탭**  
   브라우저가 서버에 어떤 요청을 보내는지 확인합니다. `Doc`은 문서 요청, `Fetch/XHR`은 JavaScript가 따로 가져오는 데이터 요청인 경우가 많습니다. 요청 URL, method(GET/POST), query string, response를 보면 코드로 요청할 주소를 찾을 수 있습니다.

개발자도구를 볼 때는 다음 질문을 순서대로 확인합니다.

- 화면에 보이는 값이 첫 HTML 응답 안에 있는가?
- HTML에 없다면 Network 탭의 `Fetch/XHR` 요청으로 따로 받아오는가?
- 페이지 번호, 검색어, 종목 코드 같은 값은 URL의 어느 부분에 들어가는가?
- 목록에서 한 건을 나타내는 반복 단위는 어떤 태그인가?
- 제목, 날짜, 링크처럼 필요한 값은 어떤 하위 태그에 들어 있는가?

개발자도구에서 CSS selector를 복사할 수도 있지만, 너무 긴 selector는 페이지가 조금만 바뀌어도 깨질 수 있습니다. `body > div > table > tbody > tr:nth-child(3)`처럼 위치에 의존하는 선택자보다 `table.type2 tr`, `td.title a.tit`처럼 의미 있는 태그와 클래스 조합을 쓰는 편이 안정적입니다.


#### 데이터 수집 원리

크롤링은 사람이 브라우저로 하는 일을 코드로 반복하는 과정입니다.

<img src="image/web_crawling_flow.svg" width="760">

이미지 출처: 김민수 강사

기본 순서는 다음과 같습니다.

1. 수집할 항목을 정합니다. 예: 뉴스 제목, 언론사, 날짜, 링크
2. `playwright codegen`으로 브라우저를 열고 실제 이동·클릭·입력을 기록합니다.
3. Playwright Inspector에서 생성된 `page.goto()`, `click()`, `fill()`, `locator()` 코드를 확인합니다.
4. 코드 초안에서 필요한 선택자만 남기고 수집 함수로 정리합니다.
5. `locator().count()`, `nth()`, `inner_text()`, `get_attribute()`로 반복되는 항목을 추출합니다.
6. 페이지 번호, 검색어, 종목 코드 같은 파라미터를 바꿔가며 반복 수집합니다.
7. 누락값, 중복, 날짜 범위를 확인한 뒤 데이터프레임이나 CSV로 저장합니다.

| 페이지 유형 | 특징 | 적합한 접근 |
|-------------|------|-------------|
| 정적 HTML 페이지 | 처음 받은 HTML 안에 필요한 데이터가 있음 | `requests + BeautifulSoup` |
| 동적 페이지 | JavaScript 실행 후 데이터가 화면에 나타남 | Network에서 API를 찾거나 `Playwright` 사용 |
| 로그인/클릭 필요 페이지 | 세션, 쿠키, 버튼 조작이 필요함 | `Playwright` |

| 도구 | 적합한 상황 | 장점 | 주의할 점 |
|------|-------------|------|-----------|
| `BeautifulSoup` | HTML 안에 데이터가 바로 들어 있는 정적 페이지 | 가볍고 코드가 단순함 | JavaScript로 나중에 그려지는 데이터에는 한계가 있음 |
| `Selenium` | 브라우저 조작 흐름을 이미 알고 있을 때 | 기존 자료와 예제가 많음 | 대기 처리와 드라이버 설정이 번거로울 수 있음 |
| `Playwright` | 최신 웹앱, 동적 렌더링, 안정적인 자동화가 필요한 페이지 | 자동 대기, 빠른 실행, codegen 지원 | 최초 브라우저 설치가 필요함 |

#### Playwright codegen으로 시작하기

Playwright는 브라우저에서 한 행동을 파이썬 코드 초안으로 만들어 줍니다. 터미널에서 실행합니다.

```bash
pip install playwright
playwright install chromium
playwright codegen "https://finance.naver.com/item/news_news.naver?code=005930&page=1&sm=title_entity_id.basic&clusterId="
```

codegen을 실행하면 브라우저와 Playwright Inspector가 함께 열립니다. 브라우저에서 페이지를 이동하거나 요소를 클릭하면 Inspector에 코드가 만들어지고, `Pick Locator`로 제목·날짜·버튼 같은 요소의 선택자를 확인할 수 있습니다.

크롤링 스크립트로 정리할 때는 codegen이 만든 테스트용 코드를 그대로 쓰기보다 다음만 남깁니다.

- `page.goto(...)`: 수집할 페이지로 이동
- `page.locator(...)`: 반복되는 항목과 필요한 하위 요소 선택
- `click()`, `fill()`: 검색, 페이지 이동처럼 사람이 하는 조작
- `wait_for_*`: 데이터가 나타날 때까지 기다리는 조건

크롤링할 때는 사이트의 `robots.txt`와 이용약관을 확인하고, 너무 빠르게 반복 요청하지 않아야 합니다. 개인정보, 저작권, 유료 콘텐츠도 함부로 수집하거나 재배포하면 안 됩니다.


#### 실습: 네이버 금융 종목 뉴스 수집

codegen으로 확인한 선택자를 수집 함수에 옮겨 봅니다. 예제에서는 네이버 금융 종목 뉴스에서 제목, 언론사, 날짜, 링크를 가져와 `news_df`를 만듭니다.

수집 대상 URL은 종목 코드와 페이지 번호만 바꾸면 됩니다.

```text
https://finance.naver.com/item/news_news.naver?code=005930&page=1&sm=title_entity_id.basic&clusterId=
```

codegen에서 확인할 핵심 선택자는 다음과 같습니다.

| 항목 | 선택자 |
|------|--------|
| 뉴스 행 | `table.type5 tr` |
| 제목 | `td.title a.tit` |
| 언론사 | `td.info` |
| 날짜 | `td.date` |


In [ ]:
import pandas as pd  # 표 형태 데이터를 다루는 라이브러리입니다.
from urllib.parse import urljoin  # 상대 경로 링크를 전체 URL로 바꾸는 도구입니다.
from playwright.async_api import async_playwright  # 주피터 노트북에서 쓰기 좋은 비동기 Playwright API입니다.

NEWS_URL_TEMPLATE = "https://finance.naver.com/item/news_news.naver?code={code}&page={page}&sm=title_entity_id.basic&clusterId="


async def collect_stock_news(code="005930", start_page=1, end_page=2):
    rows = []  # 여러 페이지의 뉴스 결과를 모읍니다.

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)  # 화면 없이 Chromium 브라우저를 실행합니다.
        page = await browser.new_page(locale="ko-KR")  # 한국어 페이지 기준으로 새 탭을 만듭니다.

        for page_no in range(start_page, end_page + 1):
            url = NEWS_URL_TEMPLATE.format(code=code, page=page_no)  # 종목 코드와 페이지 번호를 URL에 넣습니다.
            await page.goto(url, wait_until="domcontentloaded")  # HTML 구조가 준비될 때까지 기다립니다.
            await page.locator("table.type5").wait_for(timeout=10_000)  # 뉴스 표가 나타날 때까지 기다립니다.

            news_rows = page.locator("table.type5 tr")  # codegen에서 확인한 반복 행입니다.
            row_count = await news_rows.count()

            for i in range(row_count):
                row = news_rows.nth(i)
                title_links = row.locator("td.title a.tit")  # 뉴스 제목 링크입니다.

                if await title_links.count() == 0:
                    continue

                title_link = title_links.first
                source_cell = row.locator("td.info")
                date_cell = row.locator("td.date")

                href = await title_link.get_attribute("href")
                rows.append({
                    "title": (await title_link.inner_text()).strip(),
                    "source": (await source_cell.inner_text()).strip() if await source_cell.count() else None,
                    "date": (await date_cell.inner_text()).strip() if await date_cell.count() else None,
                    "link": urljoin("https://finance.naver.com", href or ""),
                    "page": page_no,
                })

            await page.wait_for_timeout(500)  # 요청 간격을 짧게 둡니다.

        await browser.close()

    news_df = pd.DataFrame(rows)
    return news_df.drop_duplicates(subset=["title", "date", "link"]).reset_index(drop=True)


news_df = await collect_stock_news("005930", start_page=1, end_page=2)  # 삼성전자 뉴스 1~2페이지를 수집합니다.
news_df.head()


위 코드는 codegen으로 확인한 선택자를 수집 함수에 옮긴 형태입니다.

- `page.goto()`는 종목 뉴스 페이지로 이동합니다.
- `page.locator("table.type5 tr")`는 뉴스 표의 행을 찾습니다.
- `td.title a.tit`, `td.info`, `td.date`는 제목, 언론사, 날짜를 꺼내는 선택자입니다.
- `wait_for()`는 표가 나타날 때까지 기다립니다. `time.sleep()`보다 화면 상태에 맞춰 기다릴 수 있습니다.
- `headless=True`를 `False`로 바꾸면 브라우저가 실제로 움직이는 모습을 볼 수 있습니다.

주피터 노트북에서는 `await collect_stock_news(...)`처럼 실행합니다. `.py` 파일로 옮길 때는 `asyncio.run()`으로 감싸면 됩니다.


### 문제. codegen으로 수집 대상 바꾸기

Playwright codegen을 다시 실행해 다른 종목 뉴스 페이지를 열어보세요. 생성된 코드에서 URL과 선택자가 어떻게 달라지는지 확인한 뒤, 아래 조건으로 `news_df`를 다시 만드세요.

조건:
- 종목 코드: `035720` (카카오)
- 수집 범위: 1~3페이지
- 결과 열: `title`, `source`, `date`, `link`, `page`

<details>
<summary>정답 보기</summary>

```python
news_df = await collect_stock_news("035720", start_page=1, end_page=3)
news_df.head()
```

</details>


### 2.2 데이터 정제 (Cleaning)
텍스트 데이터에는 사람이 쓰면서 생긴 **불필요한 잡음(noise)** 이 많이 섞여 있습니다.  
이런 잡음을 제거하지 않으면 모델이 쓸데없는 패턴까지 학습해서 성능이 떨어질 수 있습니다.  

#### 정제 대상
- 불필요한 특수문자, 구두점  
- HTML 태그  
- 중복된 공백, 줄바꿈  
- 대소문자 혼재  
- 불용어(stopwords)


####  다양한 예시

1) **특수문자 제거**
```
원문: "안녕??? 오늘 날씨 진짜 좋다~~~^^"
정제: "안녕 오늘 날씨 진짜 좋다"
```

2) **HTML 태그 제거**
```
원문: "<div>이 영화 <b>정말</b> 최고!!!</div>"
정제: "이 영화 정말 최고"
```

3) **중복 공백/개행 제거**
```
원문: "오늘은   점심에    김밥을   먹었다. \n\n 내일도 김밥?"
정제: "오늘은 점심에 김밥을 먹었다. 내일도 김밥?"
```

4) **대소문자 통일**
```
원문: "Apple is Better than apple."
정제(소문자화): "apple is better than apple."
```

5) **불용어 제거** *(전통 ML·IR에서 선택적 / **LLM 파이프라인에선 보통 사용하지 않음**)*  
- **불용어(Stopwords)**: 문장에서 자주 등장하지만 분류·검색 성능에 **한정적으로만** 기여하는 단어 집합.  
  예: 국문 — “나는/그리고/하지만/오늘/에서 …”, 영문 — “the/and/of/to …”  
- 전통 BoW/TF-IDF, IR 인덱싱에서 **차원 축소·노이즈 감소** 목적으로 **선택적** 사용.  
- LLM 파이프라인(사전학습/미세조정/추론)에서는 **보존**하는 것이 일반적.

```
원문: "나는 오늘 점심으로 김밥을 먹었다."
정제(불용어 제거 예 — 전통 ML/IR용): "오늘 점심 김밥 먹었다"
```


👉 이렇게 정제를 통해 **텍스트를 더 단순하고 의미 중심적으로 바꿔야** 이후 단계(토큰화, 임베딩 등)가 효과적으로 작동합니다.

#### 목적에 따라 전처리 전략은 달라집니다

전처리가 항상 좋은 것은 아닙니다. **무엇을 위해 사용하는가**에 따라 달라집니다.

| 목적 | 전처리 필요성 | 이유 |
|------|----------------|------|
| **전통 워드클라우드, TF-IDF 분석** | 높음 | 단순 빈도 기반이므로 노이즈 단어가 시각화를 망침 |
| **전통 ML(감성분석, 분류 등)** | 중간 | 불용어 제거·토큰 정규화가 도움되기도 함 |
| **LLM 파이프라인(사전학습·미세조정·RAG·챗봇)** | 낮음 | LLM은 문맥 기반 모델이므로 불용어·기호도 의미 구조 해석에 필요 |
| **언어학적 분석(구문·어휘 다양성)** | 매우 낮음 | 원문 보존이 핵심 |

즉, **좋은 전처리란, 모든 것을 없애는 것이 아니라 목적에 맞게 적절히 다듬는 것**입니다.

#### 실습: 전처리한 텍스트로 워드클라우드 만들기

워드클라우드는 단어 빈도를 글자 크기로 보여주는 시각화입니다.  
조사, 숫자, 특수문자 같은 노이즈를 정리한 뒤에 만들면 핵심 단어가 훨씬 잘 보입니다.

아래는 같은 방식으로 만들 수 있는 워드클라우드 예시입니다. 실제 데이터가 아니라 김민수 강사가 직접 만든 이미지라 저작권 걱정 없이 사용할 수 있습니다.

<img src="image/wordcloud_example.svg" width="760">

이미지 출처: 김민수 강사



In [ ]:
import re  # 정규표현식을 사용하기 위한 모듈입니다.
from collections import Counter  # 단어 빈도를 세는 도구입니다.

import matplotlib.pyplot as plt  # 그래프를 그리는 시각화 도구입니다.
import matplotlib.font_manager as fm  # 시스템 폰트를 찾기 위한 도구입니다.
from wordcloud import WordCloud  # 단어 빈도를 워드클라우드로 시각화합니다.

texts = [
    "배송 지연으로 고객 불만이 증가했습니다. 배송 안내가 더 필요합니다.",
    "환불 요청이 많아 상담 대기 시간이 길어졌습니다.",
    "상품 품질은 좋지만 포장 불량과 배송 지연이 반복됩니다.",
    "빠른 환불 처리와 친절한 상담이 고객 만족을 높였습니다.",
    "배송 상태 알림과 교환 절차 안내를 개선해야 합니다."
]  # 분석할 예시 문장입니다.

stopwords = {"이", "가", "을", "를", "은", "는", "과", "와", "으로", "더"}  # 제외할 단어입니다.

def clean_text(text):
    text = re.sub(r"[^가-힣a-zA-Z\s]", " ", text)  # 한글/영문/공백만 남깁니다.
    return re.sub(r"\s+", " ", text).strip()  # 여러 공백을 하나로 정리합니다.

tokens = []
for text in texts:
    cleaned = clean_text(text)  # 문장별로 노이즈를 제거합니다.
    tokens.extend([word for word in cleaned.split() if word not in stopwords and len(word) > 1])  # 의미 단어만 남깁니다.

freq = Counter(tokens)  # 단어별 등장 횟수를 계산합니다.
freq.most_common(10)  # 가장 자주 나온 단어를 확인합니다.

font_candidates = ["Noto Sans CJK KR", "NanumGothic", "Malgun Gothic", "AppleGothic"]  # 한글 폰트 후보입니다.
font_path = next(
    (font.fname for font in fm.fontManager.ttflist if any(name in font.name for name in font_candidates)),
    None
)  # 사용 가능한 한글 폰트 경로를 찾습니다.

if font_path is None:
    raise RuntimeError("한글 폰트를 찾지 못했습니다. Noto Sans CJK 또는 NanumGothic을 설치하세요.")

wc = WordCloud(
    font_path=font_path,
    width=900,
    height=450,
    background_color="white",
    colormap="viridis"
).generate_from_frequencies(freq)  # 단어 빈도로 워드클라우드를 만듭니다.

plt.figure(figsize=(10, 5))  # 그래프 크기와 도화지를 설정합니다.
plt.imshow(wc, interpolation="bilinear")  # 워드클라우드 이미지를 표시합니다.
plt.axis("off")  # 축을 숨깁니다.
plt.show()  # 그래프를 화면에 출력합니다.


#### 문제 1. 대소문자 통일
문장 `"Machine Learning is FUN and Useful."`을 모두 소문자로 바꿔보세요.  

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 필요한 값을 만들고 결과를 확인합니다.
# 문제에서 요구한 결과를 계산하고 확인합니다.

text = "Machine Learning is FUN and Useful."  # 실습할 문자열을 준비합니다.
print(text.lower())  # 결과를 화면에 출력합니다.
# 출력: "machine learning is fun and useful."

```
</details>

In [ ]:
# 여기에 작성하세요
text = "Machine Learning is FUN and Useful."

#### 문제 2. 불용어 제거
문장 `"나는 오늘 아침에 학교에 갔다."`에서 불용어 ["나는", "오늘", "에"]를 제거해보세요.  

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 필요한 값을 만들고 결과를 확인합니다.
# 문제에서 요구한 결과를 계산하고 확인합니다.

text = "나는 오늘 아침에 학교에 갔다."  # 실습할 문자열을 준비합니다.
stopwords = ["나는", "오늘", "에"]  # 분석에서 제외할 불용어 목록입니다.

for word in stopwords:  # 값을 하나씩 꺼내 반복합니다.
    text = text.replace(word, '')  # 실습할 문자열을 준비합니다.
    
text.strip()  # 양쪽 공백을 제거합니다.
print(text)  # 결과를 화면에 출력합니다.
# 출력: "아침에 학교에 갔다."

```
</details>

In [ ]:
# 여기에 작성하세요
text = "나는 오늘 아침에 학교에 갔다."
stopwords = ["나는", "오늘", "에"]

### 2.3 정규표현식(Regex)
정규표현식(Regular Expression, **regex**)은 **문자열에서 특정 패턴을 찾고/바꾸고/분리**하는 강력한 도구입니다.  
전자영수증에서 숫자만 뽑아내고, 로그에서 IP만 추출하고, 텍스트 노이즈를 빠르게 정리하는 등 **대량의 텍스트 처리 자동화**에 핵심적으로 쓰입니다.


#### 왜 Regex를 쓰나요?
- **일관된 패턴**을 한 번에 처리 (이메일, URL, 날짜, 숫자, 해시태그 등)
- **간결한 코드**로 복잡한 문자열 조작 수행
- 전처리(클리닝) 단계에서 **재사용 가능한 규칙**으로 품질 유지


#### 핵심 문법(요약)
- **문자클래스**: `\d`(숫자), `\D`(숫자 아님), `\w`(단어문자: [A-Za-z0-9_]), `\s`(공백)  
- **반복/수량자**: `*`(0+), `+`(1+), `?`(0 or 1), `{m,n}`(m~n회)  
- **그룹/선택**: `( )`(캡처 그룹), `(?: )`(비캡처), `|`(OR)  
- **앵커**: `^`(문자열/행 시작), `$`(문자열/행 끝), `\b`(단어 경계)  
- **플래그**: `re.I`(대소문자 무시), `re.M`(멀티라인: ^,$를 행 단위로), `re.S`(dot이 개행 포함)  
- **탐욕/게으름**: `.*`(탐욕적), `.*?`(게으름; 가능한 한 짧게)

> **주의**: 파이썬에서는 `r"..."` **원시 문자열**을 사용해 백슬래시 이스케이프를 피하세요.


#### 정규표현식 주요 패턴 표

| 패턴 | 의미 | 예시 | 매칭 결과 |
|------|------|------|-----------|
| `.` | 임의의 한 문자(개행 제외) | `a.c` | `abc`, `axc` |
| `^` | 문자열/행의 시작 | `^Hi` | `"Hi there"` |
| `$` | 문자열/행의 끝 | `end$` | `"the end"` |
| `\d` | 숫자 (0–9) | `\d{3}` | `123`, `007` |
| `\D` | 숫자가 아닌 문자 | `\D+` | `"abc"`, `"--"` |
| `\w` | 단어문자 `[A-Za-z0-9_]` | `\w+` | `"hello"`, `"Python3"` |
| `\W` | 단어문자가 아닌 것 | `\W+` | `"!!"`, `" "` |
| `\s` | 공백 (스페이스, 탭, 개행) | `a\sb` | `"a b"` |
| `\S` | 공백이 아닌 문자 | `\S+` | `"text"`, `"123"` |
| `*` | 0회 이상 반복 | `ab*` | `"a"`, `"ab"`, `"abbb"` |
| `+` | 1회 이상 반복 | `ab+` | `"ab"`, `"abbb"` |
| `?` | 0회 또는 1회 | `ab?` | `"a"`, `"ab"` |
| `{m,n}` | m~n회 반복 | `\d{2,4}` | `99`, `2025` |
| `( )` | 그룹화 / 캡처 | `(ab)+` | `"ab"`, `"abab"` |
| `(?: )` | 비캡처 그룹 | `(?:ab)+` | `"abab"` |
| `|` | OR 선택 | `cat|dog` | `"cat"`, `"dog"` |
| `\b` | 단어 경계 | `\bcat\b` | `"cat"` (단어 단독일 때) |
| `(?i)` | 대소문자 무시 플래그 | `(?i)abc` | `"abc"`, `"ABC"` |

#### 파이썬 정규표현식 함수 요약

| 함수 | 설명 | 예시 코드 | 결과 |
|------|------|-----------|------|
| `re.match(pattern, string)` | 문자열 **처음부터** 패턴 매칭 | `re.match(r"\d+", "123abc")` | `<Match '123'>` |
| `re.search(pattern, string)` | 문자열 전체에서 **처음 매칭되는 패턴** 찾기 | `re.search(r"\d+", "abc123xyz")` | `<Match '123'>` |
| `re.findall(pattern, string)` | **모든 매칭 결과**를 리스트로 반환 | `re.findall(r"\d+", "a12 b34 c56")` | `['12', '34', '56']` |
| `re.finditer(pattern, string)` | 모든 매칭 결과를 **이터레이터(객체)** 로 반환 | `[m.group() for m in re.finditer(r"\d+", "a12 b34")]` | `['12', '34']` |
| `re.sub(pattern, repl, string)` | 패턴을 다른 문자열로 **치환** | `re.sub(r"\d+", "#", "ID123")` | `"ID#"` |
| `re.split(pattern, string)` | 패턴 기준으로 문자열 **분리** | `re.split(r"\s+", "a b   c")` | `['a', 'b', 'c']` |
| `re.fullmatch(pattern, string)` | 문자열 전체가 패턴과 **완전히 일치**할 때 매칭 | `re.fullmatch(r"\d{3}", "123")` | `<Match '123'>` |
| `re.compile(pattern)` | 정규표현식을 객체로 컴파일 (재사용 최적화) | `p = re.compile(r"\d+")`<br>`p.findall("1a2b3")` | `['1','2','3']` |

> ⚠️ `match`는 문자열의 시작 부분만 확인, `search`는 전체 탐색을 수행한다는 점이 중요합니다.

#### 자주 쓰는 패턴 예시

In [ ]:
### 1) 숫자/기호 제거 + 공백 정규화

import re  # 정규표현식을 사용하기 위한 모듈입니다.
text = "오늘은 2025년 9월 10일!!! 날씨   정말 좋다   ^^"
no_digits = re.sub(r"\d+", "", text)               # 숫자 제거
no_punct  = re.sub(r"[^\w\s가-힣]", " ", no_digits) # 기호 제거(한글/영문/숫자/공백만 남김)
cleaned   = re.sub(r"\s+", " ", no_punct).strip()   # 다중 공백 → 단일 공백
print(cleaned)  # "오늘은 년 월 일 날씨 정말 좋다"

In [ ]:
### 2) 이메일 추출

import re  # 정규표현식을 사용하기 위한 모듈입니다.
text = "문의: admin@example.com, 혹은 support@my-site.co.kr 로 연락주세요."
emails = re.findall(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[A-Za-z]{2,}", text)
print(emails)  # ['admin@example.com', 'support@my-site.co.kr']

In [ ]:
### 3) URL 제거 (http/https)

import re  # 정규표현식을 사용하기 위한 모듈입니다.
text = "공식 문서: https://docs.python.org 참고, 우리 블로그 http://example.com/blog 도 봐요."
no_url = re.sub(r"https?://\S+", "", text).strip()  # 정규표현식으로 문자열을 치환합니다.
print(no_url)  # "공식 문서:  참고, 우리 블로그  도 봐요."

In [ ]:
### 4) 한국 전화번호 마스킹

import re  # 정규표현식을 사용하기 위한 모듈입니다.
text = "연락처: 010-1234-5678 / 02-345-6789"
masked = re.sub(r"\b(\d{2,3})-(\d{3,4})-(\d{4})\b", r"\1-****-****", text)  # 정규표현식으로 문자열을 치환합니다.
print(masked)  # "연락처: 010-****-**** / 02-****-****"

In [ ]:
### 5) 멀티라인에서 행 시작/끝 활용 (`re.M`)

import re  # 정규표현식을 사용하기 위한 모듈입니다.
log = "OK: step1\nERROR: step2 failed\nOK: step3"
errors = re.findall(r"^ERROR:.*$", log, flags=re.M)
print(errors)  # ['ERROR: step2 failed']

In [ ]:
### 6) 탐욕 vs 게으름 (HTML 태그 사이 내용 캡처 예시)

import re  # 정규표현식을 사용하기 위한 모듈입니다.
html = "<p>첫째</p><p>둘째</p>"
greedy = re.findall(r"<p>.*</p>", html)      # 탐욕적: 한 방에 다 먹음
lazy   = re.findall(r"<p>.*?</p>", html)     # 게으름: 가능한 짧게 두 개로 나눔
print(greedy)  # ['<p>첫째</p><p>둘째</p>']
print(lazy)    # ['<p>첫째</p>', '<p>둘째</p>']

> **HTML 파싱은 정규표현식만으로 완벽히 처리하기 어렵습니다.**  
> HTML 문자열은 `BeautifulSoup`, 브라우저가 그린 DOM은 Playwright `locator`처럼 구조를 읽는 도구로 다루는 편이 안정적입니다.


#### 문제 3. 숫자와 기호 제거 + 공백 정규화
문자열에서 **숫자/특수기호를 제거**하고, **다중 공백을 하나로** 바꾼 문자열을 출력하세요.  
문자/숫자/공백/한글만 남기도록 하세요.

<details> <summary>정답 보기</summary>

```python 
import re  # 정규표현식을 사용하기 위한 모듈입니다.
text = "정가: 19,800원!!! ★★ 대박 할인 30% ★★  (한정 수량)"
tmp = re.sub(r"\d+", "", text)                         # 숫자 제거
tmp = re.sub(r"[^\w\s가-힣]", " ", tmp)                # 기호 제거
result = re.sub(r"\s+", " ", tmp).strip()             # 공백 정규화
print(result)  # "정가 원 대박 할인 한정 수량"
```
</details>



In [ ]:
# 여기에 작성하세요
import re  # 정규표현식을 사용하기 위한 모듈입니다.
text = "정가: 19,800원!!! ★★ 대박 할인 30% ★★  (한정 수량)"

#### 문제 4. 이메일만 추출하기
문장에서 모든 이메일을 찾아 **리스트**로 반환하세요.

<details> <summary>정답 보기</summary>

```python 
import re  # 정규표현식을 사용하기 위한 모듈입니다.
text = "메일: kim.ai@univ.ac.kr; 홍보: sales-team@example.com; 오류: bug+test@my.io"
emails = re.findall(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[A-Za-z]{2,}", text)
print(emails)  # 문자열을 정수로 바꿉니다.
# ['kim.ai@univ.ac.kr', 'sales-team@example.com', 'bug+test@my.io']
```
</details>



In [ ]:
# 여기에 작성하세요
text = "메일: kim.ai@univ.ac.kr; 홍보: sales-team@example.com; 오류: bug+test@my.io"

#### 문제 5. URL 제거하기
문장에서 **http/https URL을 모두 제거**하세요. (도메인 뒤 공백 정리 포함)

<details> <summary>정답 보기</summary>

```python 
import re  # 정규표현식을 사용하기 위한 모듈입니다.
text = "문서: https://a.b/c?x=1  블로그: http://blog.com/post  끝."
no_url = re.sub(r"https?://\S+", "", text)     # URL 제거
no_url = re.sub(r"\s+", " ", no_url).strip()   # 공백 정리
print(no_url)  # "문서: 블로그: 끝."
```
</details>

In [ ]:
# 여기에 작성하세요
text = "문서: https://a.b/c?x=1  블로그: http://blog.com/post  끝."

#### 문제 6. 전화번호 마스킹
문장에서 한국식 전화번호(예: `010-1234-5678`, `02-345-6789`)의 **가운데/끝 4자리**를 `*`로 마스킹하세요.

<details> <summary>정답 보기</summary>

```python
# 풀이 흐름: 문자열을 다루기 쉬운 형태로 정리합니다.
# 필요한 값만 추출하거나 비교합니다.

import re  # 정규표현식을 사용하기 위한 모듈입니다.
text = "문의: 010-1234-5678 / 대리점: 031-987-6543 / 회사: 02-345-6789"  # 실습할 문자열을 준비합니다.
masked = re.sub(r"\b(\d{2,3})-(\d{3,4})-(\d{4})\b", r"\1-****-****", text)  # 정규표현식으로 문자열을 치환합니다.
print(masked)  # 결과를 화면에 출력합니다.
# "문의: 010-****-**** / 대리점: 031-****-**** / 회사: 02-****-****"

``` 
</details>



In [ ]:
# 여기에 작성하세요
text = "문의: 010-1234-5678 / 대리점: 031-987-6543 / 회사: 02-345-6789"

#### 문제 7. 문장 분리 (간단 버전)
`.` `?` `!` 중 하나로 끝나는 구두점 기준으로 **문장을 분리**하고, 출력 시 앞뒤 공백을 제거하세요.

<details> <summary>정답 보기</summary>

```python 
import re  # 정규표현식을 사용하기 위한 모듈입니다.
text = "안녕하세요! NLP 재밌죠? 지금은 정규표현식 중입니다.  예시를 더 볼까요!"
sentences = re.split(r"[.!?]+", text)  # 문자열을 기준에 따라 나눕니다.
sentences = [s.strip() for s in sentences if s.strip()]  # 양쪽 공백을 제거합니다.
print(sentences)  # 문자열을 정수로 바꿉니다.
# ['안녕하세요', 'NLP 재밌죠', '지금은 정규표현식 중입니다', '예시를 더 볼까요']
```
</details>

In [ ]:
# 여기에 작성하세요
text = "안녕하세요! NLP 재밌죠? 지금은 정규표현식 중입니다.  예시를 더 볼까요!"

### 2.4 토큰화(Tokenization)

토큰화(Tokenization)는 **텍스트를 작은 단위(토큰, token)로 분리하는 과정**입니다.  
컴퓨터는 텍스트를 숫자로 바꿔야 이해할 수 있는데, 긴 문장을 그대로 하나의 단위로 두면 너무 복잡하고 차원이 폭발합니다.  
따라서 문장을 규칙적으로 잘라 **모델이 처리 가능한 단위**로 나누는 것이 필요합니다.  



#### 왜 토큰화가 중요한가?
- 딥러닝 모델은 보통 **최대 입력 길이 제한**이 있음 → 긴 문장은 적절히 쪼개서 입력해야 함  
- 텍스트를 잘 분리해야 단어·형태·문맥 정보를 더 잘 반영할 수 있음  
- **희귀 단어 문제(OOV, Out Of Vocabulary)** 완화:  
  - 전통 단어 기반 토큰화에서는 사전에 없는 단어(OOV)가 나오면 모델이 처리 불가  
  - Subword 기반 토큰화는 단어를 더 작은 단위로 쪼개어 표현 가능 → **사전에 없는 새로운 단어도 기존 subword들의 조합으로 처리 가능**  

👉 즉, 토큰화는 단순히 “사람이 띄어쓰기 한 단위”가 아니라, **모델이 학습하기 좋은 입력 단위로 나누는 과정**입니다. 

#### 토큰을 너무 잘게 또는 너무 크게 나눌 때 생기는 문제와, Subword가 해답이 되는 이유

토큰화의 핵심은 **각 토큰이 어느 정도 ‘의미 단위’를 갖도록 나누는 것**인데,  
너무 잘게 쪼개버리면 **각 조각이 의미를 잃고 모델이 문맥을 이해하기 훨씬 어려워진다**는 문제가 있습니다.

예를 들어,

- "사랑"이라는 단어는 → **love**라는 뚜렷한 의미가 있음  
- 하지만 너무 잘게 쪼개면  
  - "사" → *사랑, 사람, 사막, 사전, 사고…* 등 무한한 단어의 시작 부분  
  - "랑" → *결코 단독 의미가 없는 조각*  
- 이렇게 되면 **개별 토큰 하나로는 의미를 유추할 수 없고**,  
  모델은 “사 + 랑 = 사랑이라는 의미”라는 사실을 **일일이 학습해서 조합**해야 함.

즉, 너무 잘게 쪼개진 토큰은  
**① 의미가 약하고,  
② 일관성도 없으며,  
③ 모델이 문맥을 스스로 재구성해야 하는 부담이 커진다.**

#### 너무 크게 쪼갤 때의 문제: 사전(Vocabulary) 크기 폭발

이번에는 반대로, 한 단어를 너무 큰 단위로(예: 띄어쓰기 단위 그대로) 토큰화했을 때의 문제를 보겠습니다.  
이 문제를 이해하려면, 먼저 **컴퓨터가 잘라낸 토큰을 어떻게 숫자로 바꾸는지** 그 과정을 알아야 합니다.

**1. 토큰화 후에는 반드시 '출석부'를 만듭니다.**
컴퓨터는 '사과', '배' 같은 글자를 직접 계산할 수 없습니다.   
그래서 잘라낸 토큰들을 모아 중복을 없애고, **고유한 번호(ID)** 를 하나씩 붙여줍니다.

- 우리가 학생들의 이름을 매번 부르는 대신 "1번, 2번" 하고 출석 번호를 부르는 것과 같습니다.
- 이 **"토큰-번호 매칭 리스트"** 를 우리는 **사전(Vocabulary)** 이라고 부릅니다.

**2. 사전(Vocabulary)의 구조**
실제 모델은 텍스트가 들어오면 이 사전을 보고 글자를 숫자로 바꿔서 받아들입니다.

> **[사전 예시]**
> - 0번: `<pad>` (빈 공간)
> - 1번: `<unk>` (모르는 단어)
> - 2번: "사랑"
> - 3번: "사람"
> - ...

**3. 정수 인코딩 (Integer Encoding)**
이제 "나는 사람이다"라는 문장이 들어오면, 컴퓨터는 사전을 참조해 다음과 같이 숫자로 변환합니다.

> "나는 사람이다"
> → 토큰화: `["나는", "사람", "이다"]`
> → 정수 변환: `[10, 3, 52]`  *(각 토큰에 해당하는 사전의 인덱스)*

**여기서 중요한 점은, 사전에 등록된 토큰의 개수(=사전의 크기)가 바로 모델이 다뤄야 할 숫자의 범위가 된다는 것입니다.**

이제, 토큰을 너무 크게(단어 단위로) 잡으면 이 **사전의 크기**에 어떤 일이 생기는지 보면:

- 예를 들어 한국어는 조사·어미 변화가 매우 다양함  
  - "먹다, 먹었다, 먹어라, 먹었습니다, 먹었겠지, 먹고, 먹고는…"  
- 이런 변형된 단어들을 **각각 하나의 토큰(하나의 인덱스)**으로 취급하면:
  - "먹다" → 인덱스 1001  
  - "먹었다" → 인덱스 1002  
  - "먹었습니다" → 인덱스 1003  
  - "먹었겠다" → 인덱스 1004  
  - … 이런 식으로 끝도 없이 늘어남
  
- 결과적으로:
  1) **사전 크기(토큰 종류 수)가 기하급수적으로 커짐**  
     - 사전에 등록해야 할 “인덱스-토큰 쌍”이 너무 많아짐  
  2) 사전이 커지면
     - 모델이 **외워야 할 토큰 종류가 너무 많아져서**  
       → 임베딩(embedding) 행렬 크기가 커지고, 메모리/학습 비용이 증가  
  3) 텍스트에 드물게 등장하는 희귀 형태(예: "먹었겠지요만")는  
     - 사전에 **한 번 밖에 안 나오는 토큰**이 되어  
       → 모델이 그 토큰의 의미를 제대로 학습하기 어려움  
  4) 사전에 등록되지 않은 완전히 새로운 단어(OOV)가 나올 수도 있음  
     - 이 경우 **그 단어 전체를 처리할 수 없음**

정리하면,  
**너무 크게 쪼개서 “단어 변형 하나당 인덱스 하나”를 부여하는 방식은**  

- 사전에 들어가야 할 토큰 종류(=인덱스 개수)가 폭발적으로 증가하고  
- 그만큼 모델의 기억해야 할 것, 학습해야 할 것, 저장해야 할 파라미터가 감당하기 힘들어지는 구조입니다.

#### 그래서 Subword가 좋은 이유: 의미 보존 + 효율성의 균형

Subword 방식은 위 두 가지 극단의 문제를 동시에 완화하는 절충안입니다.

1) **너무 잘게 쪼갤 때의 의미 손실 → 완화**  
   - "사"처럼 의미 없는 최소 단위까지 쪼개지 않음  
   - 대신, 말뭉치(코퍼스)를 분석해서 **자주 등장하고 의미 있는 단위들**을 subword로 선택  
   - 예: "사랑", "사람", "사막"이 자주 나오면,  
     - "사" 하나가 아니라 "사랑", "사람", "사막" 자체 또는 그 안의 유의미한 조각들을 토큰으로 삼음

2) **너무 크게 쪼갤 때의 사전 폭발 → 완화**  
   - “단어 단위 전체”를 무조건 하나의 토큰으로 만들지 않음  
   - 대신, 적당한 subword 단위를 사전에 등록해 두고,  
     새로운 단어가 등장하면 **기존 subword들을 조합해서 표현**  
   - 예: "자연어처리학개론"이 처음 나와도  
     - ["자연", "어", "처리", "학", "개론"]처럼 기존에 있는 조각들로 나눠서 인덱스로 변환 가능

결국 Subword 토큰화는:

- **너무 잘게** → 토큰 하나에 의미가 거의 없음 → 문맥 이해 어려움  
- **너무 크게** → 토큰 하나당 인덱스가 너무 많아짐 → 사전 폭발 & 학습 비효율  

이 두 문제 사이에서, **각 토큰이 어느 정도 의미를 가지면서도, 사전 크기를 관리 가능한 수준으로 유지**하게 해 주는 방법입니다.

그래서 현대 NLP에서는 **Subword 기반 토큰화(BPE, SentencePiece 등)** 가  
사실상 표준처럼 널리 사용되고 있습니다.

#### 직관적 비유
- 긴 텍스트는 한 번에 삼키기 어려운 **큰 빵**과 같다.  
- 토큰화는 빵을 적당히 잘라서 **한 입 크기 조각**으로 만드는 과정.  
- 조각이 너무 크면 먹기 힘들고, 너무 잘게 부수면 원래의 맛을 잃을 수 있음.  
- Subword 방식은 “적당한 크기의 조각”을 만들어, 모델이 새로운 빵(신조어, 희귀 단어)도 부담 없이 먹게 해줌.  


#### 정리
- 토큰화는 NLP 파이프라인의 출발점  
- 목적: 모델이 처리할 수 있는 단위로 텍스트를 나누기 위함  
- 단어 기반 → 단순하지만 한계 존재  
- 형태소 기반 → 한국어 같은 교착어에 유리  
- Subword 기반 → 희귀 단어·신조어 대응

#### 한국어 형태소 분석: KoNLPy

한국어는 조사와 어미가 단어에 붙어서 의미를 만듭니다.  
띄어쓰기만 기준으로 자르면 `고객이`, `고객은`, `고객에게`가 서로 다른 토큰처럼 처리될 수 있습니다.  

**형태소 분석**은 문장을 더 작은 의미 단위로 나누고, 각 단어의 품사를 함께 확인하는 방법입니다.  
KoNLPy는 한국어 형태소 분석기를 파이썬에서 사용할 수 있게 해주는 라이브러리입니다.

아래 예제는 `Okt` 분석기로 세 가지 결과를 확인합니다.

- `morphs`: 형태소 단위로 나누기
- `nouns`: 명사만 추출하기
- `pos`: 형태소와 품사 태그 함께 보기


In [ ]:
from konlpy.tag import Okt

okt = Okt()

text = "고객이 배송 지연으로 환불을 요청했습니다."

print("형태소:", okt.morphs(text))  # 문자열을 정수로 바꿉니다.
print("명사:", okt.nouns(text))  # 문자열을 정수로 바꿉니다.
print("품사:", okt.pos(text))  # 문자열을 정수로 바꿉니다.


#### 형태소 분석 결과 활용

텍스트 분류나 검색에서는 모든 조사까지 다 쓰기보다, 의미를 많이 담는 명사·동사·형용사를 남기는 방식이 자주 쓰입니다.  
아래 예시는 고객 문의 문장에서 주요 단어만 뽑는 간단한 전처리입니다.


In [ ]:
sentences = [
    "고객이 배송 지연으로 환불을 요청했습니다.",
    "상담사는 주문번호를 확인하고 처리 상태를 안내했습니다.",
    "환불 규정에 따라 영업일 기준 3일 안에 처리됩니다."
]

stopwords = {"이", "가", "을", "를", "은", "는", "으로", "하고", "에", "안에", "의"}

tokenized = []
for sentence in sentences:
    tokens = [
        word for word, tag in okt.pos(sentence, stem=True)
        if tag in ["Noun", "Verb", "Adjective"] and word not in stopwords
    ]
    tokenized.append(tokens)

tokenized


#### 문제 8. KoNLPy로 명사 추출
문장 `"품질 점검 중 센서 오류가 반복적으로 발생했습니다."`에서 명사만 추출하세요.  

<details> <summary>정답 보기</summary>

```python
from konlpy.tag import Okt

okt = Okt()
sentence = "품질 점검 중 센서 오류가 반복적으로 발생했습니다."
nouns = okt.nouns(sentence)
print(nouns)  # 문자열을 정수로 바꿉니다.
# 출력 예시: ['품질', '점검', '중', '센서', '오류']
```
</details>


In [ ]:
# 여기에 정답을 작성하세요
sentence = "품질 점검 중 센서 오류가 반복적으로 발생했습니다."


#### 실습: SentencePiece로 Subword 나누기

지금까지는 토큰화와 Subword의 개념을 직관적으로 살펴봤습니다.  
이제 SentencePiece를 사용해 실제로 Subword 사전을 만들어 보겠습니다.


#### 1) 간단한 학습용 데이터 준비

##### `text_assets/sample.txt`

아래는 학습용 말뭉치 파일(`text_assets/sample.txt`)의 일부 예시입니다.  
자연어 처리, 토큰화, 모델 학습과 관련된 문장들이 다양하게 포함되어 있습니다.

```
자연어 처리는 재미있다.
자연어 처리를 배우는 것은 유익하다.
나는 오늘도 자연어 처리를 공부한다.
...

   (중략)

...
텍스트 요약은 중요한 정보를 짧게 줄이는 기술이다.
자연어 처리는 음성 인식이나 번역에도 응용된다.
딥러닝의 발전은 자연어 처리 성능을 크게 향상시켰다.
나는 앞으로도 NLP 분야를 꾸준히 공부할 예정이다.
```

> 위와 같이 자연어 처리 관련 표현, 조사·어미 변화, 다양한 문장 구조 등이 포함된 학습용 샘플 텍스트입니다.


#### 2) SentencePiece 모델 학습

- `--model_type`은 `bpe`, `unigram`, `char`, `word` 등 선택 가능
- 여기서는 BPE(Byte Pair Encoding) 사용

In [ ]:
import sentencepiece as spm

# SentencePiece 모델 학습
spm.SentencePieceTrainer.train(
    input="text_assets/sample.txt",       # 학습에 사용할 텍스트 파일 경로
    model_prefix="text_assets/spm",       # 출력 파일 이름 prefix (spm.model, spm.vocab 생성)
    vocab_size=220,                # 생성할 토큰(서브워드) 개수. 최소 문자+메타토큰 이상으로 설정해야 함
    model_type="bpe",              # 사용할 토큰화 알고리즘: bpe / unigram / char / word 중 선택
    character_coverage=1.0         # 학습 데이터에 등장하는 문자를 100% 포함하도록 설정 (한국어는 1.0 권장)
)

#### 3) 학습된 모델 불러오기 & 토큰화

In [ ]:
import sentencepiece as spm

sp = spm.SentencePieceProcessor()
sp.load("text_assets/spm.model")

# 테스트 문장
text = "자연어처리학개론은 흥미롭다."

# 토큰화
tokens = sp.encode_as_pieces(text)
ids = sp.encode_as_ids(text)

print("입력 문장:", text)  # 문자열을 정수로 바꿉니다.
print("Subword 토큰:", tokens)  # 문자열을 정수로 바꿉니다.
print("토큰 ID:", ids)  # 문자열을 정수로 바꿉니다.

**출력 예시**
```
입력 문장: 자연어처리학개론은 흥미롭다.
Subword 토큰: ['▁자연어', '처리', '학개론', '은', '▁', '흥', '미', '롭', '다', '.']
토큰 ID: [7, 43, 0, 74, 53, 0, 68, 0, 56, 54]
```


#### 문제 9. 단어 단위 토큰화
문장 `"오늘은 자연어 처리를 공부한다."`를 **띄어쓰기 기준**으로 토큰화하세요.  

<details> <summary>정답 보기</summary>

```python
# 풀이 흐름: 문자열을 다루기 쉬운 형태로 정리합니다.
# 필요한 값만 추출하거나 비교합니다.

sentence = "오늘은 자연어 처리를 공부한다."  # 분석할 문장을 입력받습니다.
tokens = sentence.split()  # 문자열을 기준에 따라 나눕니다.
print(tokens)  # 결과를 화면에 출력합니다.
# 출력: ['오늘은', '자연어', '처리를', '공부한다.']

```
</details>

In [ ]:
# 여기에 정답을 작성하세요
sentence = "오늘은 자연어 처리를 공부한다."

#### 문제 10. Subword 토큰화의 장점
문장 `"자연어처리학개론"` 같은 긴 단어가 있을 때, Subword 토큰화의 장점을 설명하세요.  

<details> <summary>정답 보기</summary>

- 긴 복합어가 그대로 단어 사전에 있으면 **희귀 단어(OOV) 문제** 발생  
- Subword 토큰화는 긴 단어를 작은 단위로 분리하여 OOV 문제를 줄임  
- 신조어나 처음 보는 단어도 subword 조합으로 표현 가능  
</details>


### 2.5 전통적 텍스트 인코딩(Traditional Text Encoding)

과거 NLP에서는 토큰화 후, 각 단어를 **모델이 계산할 수 있는 벡터**로 변환하기 위해  
다음과 같은 **연속된 인코딩 파이프라인**을 사용했습니다.


#### 1) 단어 사전(Dictionary) 생성 + 정수 ID 부여

먼저 개발자가 직접 단어 사전을 만들고, 각 단어에 고유한 정수 ID를 부여합니다.

예:
```
{"나는": 1, "오늘": 2, "밥을": 3, "맛있게": 4, "먹었다": 5}
```

이 정수 ID는 단어를 **구분하기 위한 식별자(identifier)** 일 뿐입니다.  
하지만 중요한 점은 다음과 같습니다:

- 정수 1, 2, 3, 4, 5는 **단어의 의미적 크기나 순서를 반영하지 않음**
- ID만으로는 **벡터 연산에 바로 사용할 수 없음**
- 모델이 “ID 5 > ID 1”처럼 **잘못된 규칙성을 학습할 위험**이 있음

따라서 정수 ID를 그대로 모델 입력으로 사용할 수 없었고,  
이를 벡터로 확장하는 추가적인 인코딩 과정이 필요했습니다.



#### 2) 원-핫 인코딩(One-hot Encoding)

정수 ID를 벡터로 변환하는 전통적 방식이 바로 **원-핫 인코딩**입니다.

단어 사전 크기(vocab_size)만큼의 벡터를 만들고,  
해당 단어 ID 위치에만 1을 두고 나머지는 0으로 채웁니다.

예: vocab_size=5

| 단어     | 나는 | 오늘 | 밥을 | 맛있게 | 먹었다 |
|----------|------|------|------|--------|--------|
| 나는     | 1    | 0    | 0    | 0      | 0      |
| 오늘     | 0    | 1    | 0    | 0      | 0      |

이 방식의 특징:

- 단어 간 의미 유사성이 **전혀 반영되지 않음**  
- 벡터 길이가 vocab_size여서 **고차원**  
- 대부분 0이므로 **희소(sparse)**  

즉, 원-핫은 단어를 “하나만 1인 표시 등”으로 구분할 뿐,  
단어의 **의미를 담는 표현**이 아닙니다.

#### 3) 원-핫 벡터와 임베딩 레이어(Embedding Layer)

원-핫 벡터는 그 자체로도 모델의 입력으로 사용될 수 있습니다(예: 초기 RNN, 간단한 분류 모델).   
하지만 **희소성(Sparsity)** 과 **차원 폭발** 문제 때문에, 딥러닝에서는 이를 **밀집 벡터(Dense Vector)**   
로 변환하는 **임베딩 레이어**를 주로 사용합니다.

즉, 원-핫 벡터를 **그대로 학습 데이터로 쓰기보다는,  
임베딩 레이어를 통과시키기 위한 '재료(인덱스 스위치)'로 사용**하는 방식입니다.

##### ✔ 원-핫 벡터의 shape
```
(1 × vocab_size) : 단어(토큰) 하나일 때
(seq_len × vocab_size) : 문장 전체일 때 (seq_len: 문장의 토큰 수)
```

##### ✔ 임베딩 레이어의 shape  
임베딩 레이어는 다음과 같은 거대한 행렬입니다:

```
(vocab_size × embedding_dim)
```

예: vocab_size=10, embedding_dim=512  
→ 임베딩 행렬 shape = (10 × 512)

##### ✔ 원-핫 × 임베딩 = 토큰 임베딩 벡터

행렬 곱:
```
(seq_len × vocab_size) × (vocab_size × 512) = (seq_len × 512)
```

즉:

- 원-핫 벡터는 **임베딩 행렬의 특정 행(row)을 선택하는 스위치**
- 결과로 **토큰 하나가 512차원 벡터**로 변환됨

#### 정리: 전통 NLP의 인코딩은  
#### **정수 ID → 원-핫 → 임베딩 레이어”의 연속된 하나의 과정**

전통 방식의 전체 흐름을 요약하면 다음과 같습니다:

```
정수 ID
  ↓ (위치만 표시)
원-핫 벡터
  ↓ (행렬 곱)
임베딩 벡터 (1 × embedding_dim)
```

여기서 임베딩 벡터가 **실제로 모델이 학습하고 사용하는 표현**입니다.

#### 참고: 현대 NLP에서는 이 과정이 완전히 자동화된다

SentencePiece, WordPiece(BERT), GPT 토크나이저는

- 토큰화  
- 정수 ID 변환  
- 특수 토큰 처리  

까지 모두 자동으로 수행하며, 이 ID는 원-핫을 만들지 않고  
곧바로 **임베딩 레이어의 행(row)을 조회(lookup)** 하여 벡터를 얻습니다.

현대 방식:
```
정수 ID → 임베딩 레이어 행 선택 → (1 × embedding_dim)
```
전통 방식의 행렬 곱 연산을  
효율화한 형태라고 이해하면 됩니다.

<img src="image/embedding_lookup_vs_onehot.svg" width="760">

이미지 출처: 김민수 강사

### 💡 핵심 요약

- 전통적 방식은  
  **정수 ID → 원핫 → 임베딩 레이어**로 구성된 연속적인 인코딩 파이프라인  
- 원-핫 벡터는 “표현”이 아니라 **임베딩을 선택하는 스위치**  
- 임베딩 레이어 shape은 `(vocab_size × embedding_dim)`  
- 토큰 하나는 최종적으로 `(1 × embedding_dim)` 벡터가 되어 모델에 전달됨  
- 현대 방식은 원-핫을 생략하고 **ID로 바로 lookup**하여 효율적으로 처리